In [1]:
import random
import xml.etree.ElementTree as ET
from datetime import date, timedelta

In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# LOAD INSTANCE — opened once, used throughout
# ═════════════════════════════════════════════════════════════════════════════
tree     = ET.parse("TestInstance.xml")
root     = tree.getroot()
teams_el = root.find("Resources").find("Teams").findall("team")
n        = len(teams_el)
teams    = {int(t.attrib["id"]): t.attrib["name"] for t in teams_el}
 
print(f"Found {n} teams: {teams}")

Found 6 teams: {0: 'Team 0', 1: 'Team 1', 2: 'Team 2', 3: 'Team 3', 4: 'Team 4', 5: 'Team 5'}


In [13]:
# ═════════════════════════════════════════════════════════════════════════════
# DISTANCE MATRIX
# one way travel time in days from team i stadium to team j stadium
# symmetric: dist[i][j] = dist[j][i], diagonal = 0
# ═════════════════════════════════════════════════════════════════════════════
random.seed(42)
dist = [[0] * n for _ in range(n)]
for i in range(n):
    for j in range(i + 1, n):
        d          = random.randint(1, 3)
        dist[i][j] = d
        dist[j][i] = d
 
print("\nDistance Matrix (one way travel days):")
print("      " + " ".join(f"T{j}" for j in range(n)))
for i in range(n):
    print(f"T{i}:    " + "  ".join(str(dist[i][j]) for j in range(n)))


Distance Matrix (one way travel days):
      T0 T1 T2 T3 T4 T5
T0:    0  3  1  1  3  2
T1:    3  0  1  1  1  3
T2:    1  1  0  1  3  3
T3:    1  1  1  0  3  1
T4:    3  1  3  3  0  3
T5:    2  3  3  1  3  0


In [14]:
# ═════════════════════════════════════════════════════════════════════════════
# TEAM STRENGTHS
# fixed synthetic ratings based on hypothetical previous season standings
# ═════════════════════════════════════════════════════════════════════════════
strengths = {0: 10, 1: 8, 2: 6, 3: 5, 4: 3, 5: 1}
STRONG_THRESHOLD = 7
print(f"\nTeam Strengths: {strengths}")


Team Strengths: {0: 10, 1: 8, 2: 6, 3: 5, 4: 3, 5: 1}


In [16]:

# INITIAL SCHEDULE — DOUBLE ROUND ROBIN via circle method
# first half: single RR, second half: flip home and away
# 3 day window per slot, 4 day gap between slot windows

def circle_method(n):
    rot    = list(range(n))
    rounds = []
    for r in range(n - 1):
        rnd = [(rot[0], rot[n-1])]
        for i in range(1, n // 2):
            rnd.append((rot[i], rot[n-1-i]))
        rounds.append(rnd)
        rot = [rot[0]] + [rot[-1]] + rot[1:-1]
    return rounds
 
first_half  = circle_method(n)
second_half = [[(away, home) for home, away in rnd] for rnd in first_half]
all_rounds  = first_half + second_half

In [17]:
# verify each team plays exactly once per slot
for r_idx, rnd in enumerate(all_rounds):
    all_t = [h for h,a in rnd] + [a for h,a in rnd]
    assert len(all_t) == len(set(all_t)) == n, f"ERROR slot {r_idx}"

In [18]:
SEASON_START = date(2024, 8, 10)
WINDOW_SIZE  = 3   # 3 day window per slot, windows are consecutive with no gap
 

In [19]:
schedule = []
for slot_idx, rnd in enumerate(all_rounds):
    window_start = SEASON_START + timedelta(days=slot_idx * WINDOW_SIZE)
    window_end   = window_start + timedelta(days=WINDOW_SIZE - 1)
    for game_idx, (home, away) in enumerate(rnd):
        schedule.append({
            "slot":         slot_idx,
            "home":         home,
            "home_name":    teams[home],
            "away":         away,
            "away_name":    teams[away],
            "date":         window_start + timedelta(days=game_idx),
            "window_start": window_start,
            "window_end":   window_end
        })

In [20]:
print(f"\nINITIAL SCHEDULE (Double Round Robin — {len(all_rounds)} slots, {len(schedule)} games)")
print("=" * 65)
current_slot = -1
for g in schedule:
    if g["slot"] != current_slot:
        current_slot = g["slot"]
        half = "FIRST HALF" if g["slot"] < n-1 else "SECOND HALF"
        print(f"\nSlot {g['slot']} [{half}] | Window: {g['window_start']} to {g['window_end']}")
        print("-" * 65)
    print(f"  {g['date']}  |  {g['home_name']} (H)  vs  {g['away_name']} (A)")
 
print(f"\nSeason: {schedule[0]['date']} to {schedule[-1]['date']}")
 


INITIAL SCHEDULE (Double Round Robin — 10 slots, 30 games)

Slot 0 [FIRST HALF] | Window: 2024-08-10 to 2024-08-12
-----------------------------------------------------------------
  2024-08-10  |  Team 0 (H)  vs  Team 5 (A)
  2024-08-11  |  Team 1 (H)  vs  Team 4 (A)
  2024-08-12  |  Team 2 (H)  vs  Team 3 (A)

Slot 1 [FIRST HALF] | Window: 2024-08-13 to 2024-08-15
-----------------------------------------------------------------
  2024-08-13  |  Team 0 (H)  vs  Team 4 (A)
  2024-08-14  |  Team 5 (H)  vs  Team 3 (A)
  2024-08-15  |  Team 1 (H)  vs  Team 2 (A)

Slot 2 [FIRST HALF] | Window: 2024-08-16 to 2024-08-18
-----------------------------------------------------------------
  2024-08-16  |  Team 0 (H)  vs  Team 3 (A)
  2024-08-17  |  Team 4 (H)  vs  Team 2 (A)
  2024-08-18  |  Team 5 (H)  vs  Team 1 (A)

Slot 3 [FIRST HALF] | Window: 2024-08-19 to 2024-08-21
-----------------------------------------------------------------
  2024-08-19  |  Team 0 (H)  vs  Team 2 (A)
  2024-08-20

In [21]:
# CONSTRAINT CHECKER
# H1: every team plays exactly once per slot, no idle slots
# H2: CA1 hard capacity constraints from ITC instance
# H3: travel feasibility — gap between consecutive games >= travel time needed
# H4: double RR integrity — each ordered fixture appears exactly once
# H5: home away balance — cumulative difference <= 1 at any slot
# S5: rest day — at least 1 rest day between consecutive games
# S6: fixture difficulty spacing — no concentration of strong opponents

In [22]:
slots         = sorted(set(g["slot"] for g in schedule))
total_penalty = 0
report        = []
 
def log(constraint, violation, amount, penalty):
    report.append({
        "constraint": constraint,
        "violation":  violation,
        "amount":     amount,
        "penalty":    penalty
    })

In [23]:
# ── H1: Round Balance and Compactness ────────────────────────────────────────
# every team plays exactly once per slot with no idle rounds
h1_v, h1_p = 0, 0
for s in slots:
    sg    = [g for g in schedule if g["slot"] == s]
    all_t = [g["home"] for g in sg] + [g["away"] for g in sg]
    for tid in teams:
        count = all_t.count(tid)
        if count == 0:
            h1_v += 1
            h1_p += 999
            log("H1", f"Team {teams[tid]} is idle in slot {s}", 1, 999)
        elif count > 1:
            h1_v += 1
            h1_p += 999
            log("H1", f"Team {teams[tid]} plays {count} times in slot {s}", count-1, 999)
total_penalty += h1_p

In [24]:
# ── H2: CA1 Hard from ITC instance ───────────────────────────────────────────
# reads hard capacity constraints from the loaded XML instance
# restricts home or away games within specified slot sets
h2_v, h2_p  = 0, 0
constraints  = root.find("Constraints")
hard_ca1     = []
if constraints is not None:
    cap = constraints.find("CapacityConstraints")
    if cap is not None:
        for c in cap.findall("CA1"):
            if c.attrib["type"] == "HARD":
                hard_ca1.append({
                    "teams":   [int(x) for x in c.attrib["teams"].split(";")],
                    "slots":   [int(x) for x in c.attrib["slots"].split(";")],
                    "max":     int(c.attrib["max"]),
                    "mode":    c.attrib["mode"],
                    "penalty": int(c.attrib["penalty"])
                })
 

In [26]:
for c in hard_ca1:
    for tid in c["teams"]:
        count = sum(
            1 for g in schedule if g["slot"] in c["slots"] and (
                (c["mode"] == "H"  and g["home"] == tid) or
                (c["mode"] == "A"  and g["away"] == tid) or
                (c["mode"] == "HA" and (g["home"] == tid or g["away"] == tid))
            )
        )
        if count > c["max"]:
            excess = count - c["max"]
            h2_v  += excess
            h2_p  += excess * 999
            log("H2",
                f"Team {teams[tid]}: {count} {c['mode']} games in slots {c['slots']}, max={c['max']}",
                excess, excess * 999)
total_penalty += h2_p

In [27]:
# ── H3: Travel Feasibility ────────────────────────────────────────────────────
# gap between consecutive games for each team must cover:
# (return journey home after g1 if away) + (travel to g2 if away)
h3_v, h3_p = 0, 0
for tid in teams:
    tg = sorted([g for g in schedule if g["home"] == tid or g["away"] == tid],
                key=lambda g: g["date"])
    for i in range(len(tg) - 1):
        g1, g2       = tg[i], tg[i+1]
        gap          = (g2["date"] - g1["date"]).days
        return_home  = dist[g1["home"]][tid] if g1["away"] == tid else 0
        travel_to_g2 = dist[g2["home"]][tid] if g2["away"] == tid else 0
        time_needed  = return_home + travel_to_g2
        if gap < time_needed:
            shortage = time_needed - gap
            h3_v    += 1
            h3_p    += 999
            log("H3",
                f"Team {teams[tid]}: {g1['date']} -> {g2['date']} | "
                f"gap={gap}d, return_home={return_home}d, "
                f"travel_to_next={travel_to_g2}d, needed={time_needed}d, short by {shortage}d",
                shortage, 999)
total_penalty += h3_p

In [28]:
# ── H4: Double RR Integrity ───────────────────────────────────────────────────
# each ordered fixture (i home, j away) must appear exactly once
h4_v, h4_p = 0, 0
for i in teams:
    for j in teams:
        if i != j:
            count = sum(1 for g in schedule if g["home"] == i and g["away"] == j)
            if count != 1:
                h4_v += 1
                h4_p += 999
                log("H4",
                    f"{teams[i]} (H) vs {teams[j]} (A) appears {count} times, expected 1",
                    abs(count-1), 999)
total_penalty += h4_p

In [29]:
# ── H5: Home Away Balance ─────────────────────────────────────────────────────
# at any slot s, cumulative home games minus cumulative away games for any team
# must not differ by more than 1
# ensures no team is disproportionately burdened with away games early in season
h5_v, h5_p = 0, 0
for tid in teams:
    for s in slots:
        cumulative_home = sum(
            1 for g in schedule if g["slot"] <= s and g["home"] == tid
        )
        cumulative_away = sum(
            1 for g in schedule if g["slot"] <= s and g["away"] == tid
        )
        diff = abs(cumulative_home - cumulative_away)
        if diff > 1:
            h5_v += 1
            h5_p += 999
            log("H5",
                f"Team {teams[tid]} at slot {s}: "
                f"home={cumulative_home}, away={cumulative_away}, diff={diff}",
                diff - 1, 999)
total_penalty += h5_p
 

In [30]:
# ── S5: Rest Day ──────────────────────────────────────────────────────────────
# each team should have at least 1 rest day between consecutive games
# excluding travel time, penalised per day short
s5_v, s5_p = 0, 0
for tid in teams:
    tg = sorted([g for g in schedule if g["home"] == tid or g["away"] == tid],
                key=lambda g: g["date"])
    for i in range(len(tg) - 1):
        gap = (tg[i+1]["date"] - tg[i]["date"]).days
        if gap < 1:
            shortage = 1 - gap
            s5_v    += 1
            s5_p    += shortage
            log("S5",
                f"Team {teams[tid]}: {tg[i]['date']} -> {tg[i+1]['date']} | "
                f"gap={gap}d, short by {shortage}d",
                shortage, shortage)
total_penalty += s5_p

In [31]:
# ── S6: Fixture Difficulty Spacing ────────────────────────────────────────────
# cumulative opponent strength in any 3 consecutive slots must not exceed
# 2 x average team strength to prevent concentration of tough fixtures
s6_v, s6_p = 0, 0
K_WINDOW    = 3
AVG_STR     = sum(strengths.values()) / n
DELTA_MAX   = 2 * AVG_STR
 
for tid in teams:
    for s in slots:
        window_slots = [x for x in slots if s <= x <= s + K_WINDOW - 1]
        if len(window_slots) < K_WINDOW:
            continue
        total_str = sum(
            strengths[g["away"] if g["home"] == tid else g["home"]]
            for g in schedule
            if g["slot"] in window_slots and (g["home"] == tid or g["away"] == tid)
        )
        if total_str > DELTA_MAX:
            excess = total_str - DELTA_MAX
            s6_v  += 1
            s6_p  += excess * 5
            log("S6",
                f"Team {teams[tid]}: slots {window_slots} | "
                f"strength={total_str:.1f}, max={DELTA_MAX:.1f}, excess={excess:.1f}",
                excess, excess * 5)
total_penalty += s6_p

In [32]:
# ═════════════════════════════════════════════════════════════════════════════
# REPORT
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("CONSTRAINT VIOLATION REPORT")
print("=" * 65)
print(f"H1 Round Balance & Compactness | violations: {h1_v} | penalty: {h1_p}")
print(f"H2 CA1 Hard                    | violations: {h2_v} | penalty: {h2_p}")
print(f"H3 Travel Feasibility          | violations: {h3_v} | penalty: {h3_p}")
print(f"H4 Double RR Integrity         | violations: {h4_v} | penalty: {h4_p}")
print(f"H5 Home Away Balance           | violations: {h5_v} | penalty: {h5_p}")
print(f"S5 Rest Day                    | violations: {s5_v} | penalty: {s5_p}")
print(f"S6 Difficulty Spacing          | violations: {s6_v} | penalty: {s6_p:.1f}")
print("-" * 65)


CONSTRAINT VIOLATION REPORT
H1 Round Balance & Compactness | violations: 0 | penalty: 0
H2 CA1 Hard                    | violations: 5 | penalty: 4995
H3 Travel Feasibility          | violations: 12 | penalty: 11988
H4 Double RR Integrity         | violations: 0 | penalty: 0
H5 Home Away Balance           | violations: 19 | penalty: 18981
S5 Rest Day                    | violations: 0 | penalty: 0
S6 Difficulty Spacing          | violations: 43 | penalty: 1355.0
-----------------------------------------------------------------


In [33]:
if report:
    print("\nDETAILED VIOLATIONS:")
    for r in report:
        print(f"  [{r['constraint']}] {r['violation']}")
        print(f"        Amount: {r['amount']} | Penalty: {r['penalty']}")
else:
    print("No violations.")
 
print("-" * 65)
print(f"TOTAL BASELINE PENALTY: {total_penalty:.1f}")
print("=" * 65)
 


DETAILED VIOLATIONS:
  [H2] Team Team 3: 2 H games in slots [2, 4, 7, 9], max=1
        Amount: 1 | Penalty: 999
  [H2] Team Team 2: 2 H games in slots [7, 0], max=1
        Amount: 1 | Penalty: 999
  [H2] Team Team 2: 2 A games in slots [1, 4, 5], max=1
        Amount: 1 | Penalty: 999
  [H2] Team Team 2: 1 A games in slots [2], max=0
        Amount: 1 | Penalty: 999
  [H2] Team Team 2: 1 A games in slots [2], max=0
        Amount: 1 | Penalty: 999
  [H3] Team Team 0: 2024-08-25 -> 2024-08-28 | gap=3d, return_home=2d, travel_to_next=3d, needed=5d, short by 2d
        Amount: 2 | Penalty: 999
  [H3] Team Team 0: 2024-08-28 -> 2024-08-31 | gap=3d, return_home=3d, travel_to_next=1d, needed=4d, short by 1d
        Amount: 1 | Penalty: 999
  [H3] Team Team 0: 2024-09-03 -> 2024-09-06 | gap=3d, return_home=1d, travel_to_next=3d, needed=4d, short by 1d
        Amount: 1 | Penalty: 999
  [H3] Team Team 1: 2024-08-18 -> 2024-08-20 | gap=2d, return_home=3d, travel_to_next=1d, needed=4d, short 

In [34]:
# ═════════════════════════════════════════════════════════════════════════════
# SAVE TO XML
# writes teams, distance matrix, strengths, and initial schedule into one file
# ═════════════════════════════════════════════════════════════════════════════
 
# add distance matrix to XML
matrix_el = ET.SubElement(root, "TravelMatrix", unit="days")
for i in range(n):
    for j in range(n):
        if i != j:
            ET.SubElement(matrix_el, "Travel",
                          home_team=str(i), away_team=str(j), days=str(dist[i][j]))
 

In [35]:
# add team strengths to XML
strengths_el = ET.SubElement(root, "TeamStrengths",
                              note="Synthetic ratings based on hypothetical previous season standings")
for tid, s in strengths.items():
    ET.SubElement(strengths_el, "Team", id=str(tid), name=teams[tid], strength=str(s))
 
# add initial schedule to XML
sched_el = ET.SubElement(root, "InitialSchedule",
                          note="Double RR via circle method. Feasible starting point.")
for g in schedule:
    ET.SubElement(sched_el, "Game",
                  slot=str(g["slot"]),
                  home=str(g["home"]),      home_name=g["home_name"],
                  away=str(g["away"]),      away_name=g["away_name"],
                  date=str(g["date"]),
                  window_start=str(g["window_start"]),
                  window_end=str(g["window_end"]))
 

In [36]:
from xml.dom import minidom
xml_str = minidom.parseString(
    ET.tostring(root, encoding="unicode")
).toprettyxml(indent="  ")

In [37]:
out = "Full_Instance.xml"
with open(out, "w") as f:
    f.write(xml_str)
 
print(f"\nSaved everything to: {out}")


Saved everything to: Full_Instance.xml
